# Linear Regression with Statsmodels


In the first notebooks of this repository, you trained linear regression models with `scikit-learn`. This notebook introduces a second library for the same job: `statsmodels`.

Both libraries fit the same ordinary least squares model and return identical coefficients. What differs is what each one is built to help you do:

| | scikit-learn | statsmodels |
|---|---|---|
| Question it answers | How well can I predict new data? | Which features matter, and by how much? |
| Typical output | Predictions, and scores on a test set | Coefficients with standard errors, p-values, confidence intervals and diagnostic tests |
| Workflow | Train/test split, pipelines, cross-validation, hyperparameter tuning | Summary tables, hypothesis tests, residual tests |
| Range of models | Many models for regression, classification, clustering and more | Statistical and econometric models, including time series |
| Typical use in a data science job | Models that go into a product or a forecast | Explaining a relationship to stakeholders, A/B tests, reports |

> [!IMPORTANT]
> Use **scikit-learn** when the goal is **prediction**, and **statsmodels** when the goal is **explanation**. Many projects use both: statsmodels to understand the data, scikit-learn to build the model that ships.

From the next lesson onwards, the course builds on scikit-learn (train/test split, regularisation, pipelines, cross-validation), because our focus is prediction.

## Learning Objectives

At the end of this notebook, you should be able to:

- Train a linear regression model with statsmodels, adding the constant for the intercept yourself.
- Extract the coefficients, $R^2$ and p-values from a statsmodels summary.
- Choose between statsmodels and scikit-learn based on whether the goal is inference or prediction.

## The Cars Dataset



We use the cars dataset from the previous notebooks to train a simple and a multiple linear regression with statsmodels.

In [ ]:
# Import packages
import pandas as pd
import statsmodels.api as sm

In [ ]:
# Using the same car dataset as before
cars = pd.read_csv("data/cars_multivariate.csv")
cars.head()

## Simple Linear Regression


For the simple linear regression, we model the target variable `mpg` with the feature `horsepower`.

In [ ]:
# Plot relationship between horsepower and mpg
cars.plot("horsepower", "mpg", kind="scatter");

As usual, prepare the data by defining the target and the feature(s).

Unlike `scikit-learn`, `statsmodels` does **not** add a constant term for the intercept by default:

- In [sklearn.linear_model.LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), the parameter `fit_intercept` defaults to `True`, so the intercept is fitted automatically.
- In [statsmodels OLS](https://www.statsmodels.org/dev/generated/statsmodels.regression.linear_model.OLS.html), you must add the constant term yourself (with `sm.add_constant()`) if you want the model to have an intercept.

Documentation: [`OLS`](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html), [`add_constant`](https://www.statsmodels.org/stable/generated/statsmodels.tools.tools.add_constant.html).

In [ ]:
# Define target and feature and add constant term
X = cars.horsepower
X = sm.add_constant(X)
y = cars.mpg

X.head()

In [ ]:
# Fit model and print summary
model = sm.OLS(y, X)
results = model.fit()

# Print intercept and coefficients
results.params

The syntax is slightly different, but the results are the same as with scikit-learn.

What makes `statsmodels` so useful for statistical analysis is the `.summary()` method, which you can call on every fitted model. It gives an overview of almost every metric and statistic you might need. At first, this can feel overwhelming.

In [ ]:
# Print extensive summary
results.summary()

### Model Interpretation



We only discuss this summary briefly here. For a fuller explanation of every metric in the summary, see the reference section at the end of the notebook.

<center><img src="assets/ols_summary.png" height="400"/></center>

* The blue part of the table gives details about the data and the model itself.
* The orange part shows the goodness of fit:
    * **$R^2$**
    * **adjusted $R^2$**
* The green part gives information about the estimated coefficients of the model:
    * **coef** contains the value of the intercept (the coefficient of the constant) and the coefficient of each feature.
    * **P>|t|** is one of the most important statistics in the summary. It is the p-value for the null hypothesis that the coefficient is zero: a small value means data like ours would be unlikely if the feature had no effect at all. See the caveats below before you rely on it.
* The turquoise part at the bottom gives information about the residuals, autocorrelation and multicollinearity.

## Multiple Linear Regression



You can also use `statsmodels` to train a multiple linear regression with several features.

In [ ]:
# Define target and features and add a constant term
X2 = cars[["horsepower", "weight"]]
X2 = sm.add_constant(X2)
y2 = cars.mpg

X2.head(n=2)

In [ ]:
# We can also combine all steps using dot notation
sm.OLS(y2, X2).fit().summary()

### Predicting for new observations



Like `scikit-learn`, `statsmodels` has a `.predict()` method to calculate predictions for new observations. Say we want to predict the `mpg` of a new car with a `horsepower` of 250 and a `weight` of 3000.

You can pass the new data as a list, but you must add a constant of 1 as the first value, for the intercept.

In [ ]:
# Make predictions on test car
test_car = [1, 250, 3000]
sm.OLS(y2, X2).fit().predict(test_car)

According to the model, a car with these characteristics would have a fuel efficiency of ~15.9 mpg.

### Try It

Add `acceleration` to the multiple regression model above. Build the features with `sm.add_constant(cars[["horsepower", "weight", "acceleration"]])`, fit the model with `sm.OLS` and print the summary. Does $R^2$ go up? Which coefficient has a p-value above 0.05, and what does that tell you?

In [ ]:
# TODO: select horsepower, weight and acceleration, and add the constant
X3 = ...

# TODO: fit sm.OLS with the target y2 and the features X3, then print the summary
results3 = ...
results3.summary()

<details><summary>Show answer</summary>

$R^2$ stays at 0.706, so `acceleration` explains no extra variance. Its p-value is 0.987: once `horsepower` and `weight` are in the model, the data gives no evidence that `acceleration` has an effect. `horsepower` (p = 0.003) and `weight` (p < 0.001) stay significant.

</details>

## Caveats: Reading a statsmodels Summary

The summary table is powerful, and it is easy to read too much into it. Keep these points in mind:

- **Add the constant yourself.** `sm.OLS` fits a line through the origin unless you call `sm.add_constant()`.
- **A p-value is not an effect size.** A tiny p-value says a coefficient is unlikely to be exactly zero, not that the feature matters much. With many observations, even a negligible effect becomes "significant".
- **The p-values rely on the model's assumptions.** If the residuals are not independent or their variance is not constant (the optional notebook 07 shows how to check this), the standard errors and p-values in the table are unreliable.
- **Testing many features produces false positives.** At the 0.05 level, about 1 in 20 features with no real effect will still look significant.
- **The $R^2$ is measured on the training data.** It says how well the model fits the data it has seen, not how well it predicts new data. From the next lesson onwards you will evaluate models on a separate test set.
- **Correlated features blur the coefficients.** With multicollinearity, individual coefficients and their p-values can change a lot when you add or remove a feature, even though the predictions barely change.

This is why the exercises in this course use scikit-learn: the focus is on prediction, measured on data the model has not seen. Use statsmodels when you want to describe and test relationships, and read its summary with these caveats in mind.

## The Model Summary Explained

Use this section as a reference: open a part when you need it. You do not need to know every statistic in the table now.

<details><summary><b>The blue part of the table gives some specifics on the data and the model</b></summary>

* **Dep. Variable**: The dependent variable (target).
* **Model**: The technique used, in abbreviated form.
* **Method**: The loss function optimised when the parameters are selected. Least Squares, because it picks the parameters that minimise the squared training error. This is closely related to the Mean Squared Error (MSE).
* **No. Observations**: The number of observations used to train the model (the size of the training data).
* **Df Residuals**: The degrees of freedom of the residuals: the number of observations minus the number of parameters (the intercept counts as a parameter). Every parameter the model estimates uses up one degree of freedom, so the degrees of freedom show how many observations remain to estimate the error once the parameters are fixed.
* **Df Model**: The degrees of freedom of the model: the number of parameters, not counting the constant (intercept) term if present. This equals the number of features.
* **Covariance Type**: The method used to estimate the covariance of the coefficients, and therefore their standard errors. `nonrobust` is the classical method, which assumes that the residuals have a constant variance. Robust options (for example `HC3`) give standard errors that stay valid when that assumption fails; they change the standard errors and p-values, not the coefficients.

</details>

<details><summary><b>The orange part of the table shows the goodness of fit</b></summary>

* **R-squared**: The coefficient of determination. It is the proportion of the variance in the target variable that the model explains. The remaining part is the variance left to the error term, the part that the model and its predictors fail to capture.
* **Adj. R-squared**: A version of R-squared that penalises additional independent variables.
* **F-statistic**: A test of whether the model as a whole explains anything: the variance explained by the model, per feature, divided by the unexplained variance, per remaining degree of freedom (the regression mean square divided by the residual mean square). It is used to calculate the p-value below.
* **Prob (F-statistic) or P-Value**: The p-value of the F-test. Its null hypothesis is that all the slope coefficients are zero, so a small value means the features together explain more than a model with only the intercept. It does not measure the size of an effect.
* **Log-likelihood**: The logarithm of the likelihood function, which measures how likely the observed data are under the model. Higher log-likelihood values indicate a better fit of the model to the data.
* **AIC**: The Akaike Information Criterion. It combines the log-likelihood with a penalty for the number of parameters, so it rewards a good fit but penalises extra independent variables. Lower is better, and it is used to compare models fitted to the same data.
* **BIC**: The Bayesian Information Criterion. Similar to the AIC, but with a higher penalty for models with more parameters.

</details>

<details><summary><b>The green second table shows the coefficient report</b></summary>

* **coef**: The estimated value of the coefficient: the amount by which the model multiplies the feature.
* **std err**: The standard error of the coefficient estimate: how much the coefficient would vary from sample to sample. A coefficient that is large compared with its standard error is estimated precisely.
* **t**: The t-statistic. It measures how statistically significant the coefficient is.
* **P > |t|**: The p-value for the null hypothesis that the coefficient is 0. If it is less than the significance level, often 0.05, there is a statistically significant relationship between the feature and the target.
* **[95.0% Conf. Interval]**: The lower and upper bounds of the 95% confidence interval: a range of plausible values for the coefficient.

</details>

<details><summary><b>The turquoise third table provides information about residuals, autocorrelation, and multicollinearity</b></summary>

* **Skewness**: A measure of the symmetry of the data about the mean. Normally distributed errors should be symmetric about the mean (equal amounts above and below the line). The normal distribution has a skew of 0.
* **Kurtosis**: A measure of the shape of the distribution. It compares the amount of data close to the mean with the amount far away from the mean (in the tails), so it describes how "peaked" the distribution is. The normal distribution has a kurtosis of 3, and the greater the number, the more the curve peaks.
* **Omnibus D'Agostino's test**: A combined statistical test for the presence of skewness and kurtosis.
* **Prob(Omnibus)**: The above statistic turned into a probability.
* **Jarque-Bera**: A different test of the skewness and kurtosis.
* **Prob (JB)**: The above statistic turned into a probability.
* **Durbin-Watson**: A test for autocorrelation in the residuals (errors that are not independent), which is often important in time-series analysis. A value close to 2 means no autocorrelation; as a rule of thumb, values between 1.5 and 2.5 are fine.
* **Cond. No**: The condition number, a test for multicollinearity (whether, in a fit with multiple parameters, the parameters are related to each other). A high condition number strongly suggests multicollinearity.

</details>

## Summary

In this notebook you:

- Trained simple and multiple linear regression models with statsmodels, using `sm.OLS` with an added constant.
- Extracted the coefficients, $R^2$ and p-values from the model summary, and listed the caveats that come with them.
- Chose between statsmodels, built for describing and testing relationships, and scikit-learn, built for prediction.